In [3]:
import cv2
import mediapipe as mp
import numpy as np
from IPython.display import clear_output
import matplotlib.pyplot as plt
import os

WINDOW_WIDTH = 320
WINDOW_HEIGHT = 240
MAX_FRAMES = 500

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=1
)


def is_hand_below_chin(hand_landmarks, face_landmarks, image_height):
    """
    Returns True if the hand is below the chin.
    hand_landmarks: MediaPipe hand landmarks
    face_landmarks: MediaPipe face landmarks
    image_height: height of the frame (to convert normalized y)
    """
    # Get chin y position (landmark 152 in face mesh)
    chin_y = face_landmarks.landmark[152].y * image_height
    
    # Get wrist y position (landmark 0 in hand)
    wrist_y = hand_landmarks.landmark[0].y * image_height
    
    return wrist_y > chin_y



    
if not os.path.exists('pushpa.png') or not os.path.exists('girlbhaai.png') :
    raise FileNotFoundError("Make sure 'pushpa.png' and 'girlbhaai.png' are in the notebook directory!")

pushpa_img = cv2.resize(cv2.imread('pushpa.png'), (WINDOW_WIDTH, WINDOW_HEIGHT))
girl_img = cv2.resize(cv2.imread('girlbhaai.png'), (WINDOW_WIDTH, WINDOW_HEIGHT))
cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, WINDOW_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, WINDOW_HEIGHT)

hands = mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


try:
    for _ in range(MAX_FRAMES):
        ret, frame = cap.read()
        if not ret:
            print("Could not read frame from webcam.")
            break

  
        frame = cv2.flip(frame, 1)  # mirror image
        frame = cv2.resize(frame, (WINDOW_WIDTH, WINDOW_HEIGHT))
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_height = frame.shape[0]

        hand_results = hands.process(rgb_frame)
        face_results = face_mesh.process(rgb_frame)

        annotated_frame = frame.copy()

        if hand_results.multi_hand_landmarks and face_results.multi_face_landmarks:
            face_landmarks = face_results.multi_face_landmarks[0]
            for hand_landmarks in  hand_results.multi_hand_landmarks:
                # Draw landmarks
                mp_drawing.draw_landmarks(
                    annotated_frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )

                if is_hand_below_chin(hand_landmarks, face_landmarks, image_height):
                    current_meme=pushpa_img.copy()
                    cv2.putText(frame, "Pushpa", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
                else:

                    current_meme = girl_img.copy()
                    
        else:
            current_meme = girl_img.copy()
            cv2.putText(frame, "Place hand under chin", (10, 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
           

                

                # Example: print index finger tip coordinates
               
              
        # ------------------------------
        # Display inline in notebook
        # ------------------------------
        combined = np.hstack((frame, current_meme))
        combined_rgb = cv2.cvtColor(combined, cv2.COLOR_BGR2RGB)
           
    
        clear_output(wait=True)
        plt.imshow(combined_rgb)
        plt.axis('off')
        plt.show()           

finally:
    cap.release()
    face_mesh.close()
    clear_output()
    print("Hand Detection Stopped")


Hand Detection Stopped


KeyboardInterrupt: 